# Module 1: E-Commerce Data Cleaning, Preprocessing & Deep EDA
### Author: Senior Data Scientist & ML Engineer Portfolio
---
### Objectives:
1. **Data Ingestion**: Load the UCI Online Retail dataset (541k+ transactional rows).
2. **Data Cleaning & Auditing**: Handle missing values, filter non-product records, and separate cancellations.
3. **Dual Dataset Architecture**:
   - Store-level cleaned dataset (for aggregate macro sales forecasting).
   - Customer-centric dataset (for RFM, churn, and collaborative filtering).
4. **Exploratory Data Analysis (EDA)**: Executive KPIs, Pareto principle in product revenues, temporal purchase rhythms, and country concentration.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Environment libraries successfully loaded.")


## 1. Data Ingestion & Schema Inspection
We load the preprocessed CSV version of the raw dataset. Each row is an invoice item representing an individual purchase transaction.


In [ ]:
# Load raw dataset
data_path = Path('../data/raw/online_retail.csv')
if not data_path.exists():
    raise FileNotFoundError("Raw CSV not found. Please run src/data/ingest.py first.")

df_raw = pd.read_csv(
    data_path,
    dtype={'InvoiceNo': str, 'StockCode': str, 'CustomerID': str},
    parse_dates=['InvoiceDate']
)

print(f"Total Raw Records: {len(df_raw):,}")
print(f"Total Features: {df_raw.shape[1]}")
df_raw.head()


## 2. Missing Value & Anomaly Diagnostics
Real transactional data contains critical gaps. We analyze null counts and assess business reasons behind missing customer identifiers (Guest Checkouts).


In [ ]:
missing_summary = pd.DataFrame({
    'Missing_Count': df_raw.isnull().sum(),
    'Percentage (%)': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
})
missing_summary[missing_summary['Missing_Count'] > 0]


## 3. Handling Cancellations & Negative Quantities
In the UK Online Retail dataset, invoice numbers prefixed with **'C'** denote customer cancellations or returns.
These rows typically carry negative quantities.


In [ ]:
cancellations = df_raw[df_raw['InvoiceNo'].str.startswith('C', na=False) | (df_raw['Quantity'] < 0)]
print(f"Total Cancellation / Return Records: {len(cancellations):,} ({len(cancellations)/len(df_raw)*100:.2f}%)")
cancellations.head(3)


## 4. Pipeline Execution: Creating Dual-Cleaned Datasets
We invoke our production cleaner from `src.data.clean.ECommerceDataCleaner` to produce:
1. `online_retail_cleaned.csv`: Validated store transactions.
2. `online_retail_customer_centric.csv`: Confirmed customer-identified transactions.


In [ ]:
import sys
sys.path.append('..')
from src.data.clean import ECommerceDataCleaner

cleaner = ECommerceDataCleaner(raw_csv_path=Path('../data/raw/online_retail.csv'))
cleaner.clean_transactions()
audit = cleaner.generate_audit_summary()

for metric, val in audit.items():
    print(f"{metric:25s}: {val}")


## 5. Executive Business KPIs
High-level operational health indicators derived from clean transactional records.


In [ ]:
from src.data.eda import ECommerceEDA

eda = ECommerceEDA(data_path=Path('../data/processed/online_retail_cleaned.csv'))
kpis = eda.compute_executive_kpis()

pd.DataFrame(list(kpis.items()), columns=['Business KPI', 'Value'])


## 6. Product Performance & Pareto Principle (80/20 Rule)
Which top products drive the greatest revenue and volume?


In [ ]:
top_products = eda.get_top_selling_products(10)
plt.figure(figsize=(10, 5))
sns.barplot(data=top_products, x='TotalRevenue', y='Description', palette='viridis')
plt.title('Top 10 Products by Gross Revenue (£)', fontsize=14, weight='bold')
plt.xlabel('Gross Revenue (£)')
plt.ylabel('Product Description')
plt.tight_layout()
plt.show()


## 7. Temporal Sales Trends & Seasonality
Analyzing monthly revenue trajectories to identify Q4 holiday shopping peaks.


In [ ]:
monthly_sales = eda.get_monthly_revenue_trend()
plt.figure(figsize=(12, 5))
plt.plot(monthly_sales['YearMonth'], monthly_sales['MonthlyRevenue'], marker='o', linewidth=2.5, color='#2b5c8f')
plt.fill_between(range(len(monthly_sales)), monthly_sales['MonthlyRevenue'], color='#2b5c8f', alpha=0.15)
plt.title('Monthly Gross Revenue Trajectory (Nov 2010 - Dec 2011)', fontsize=14, weight='bold')
plt.xlabel('Year-Month')
plt.ylabel('Gross Revenue (£)')
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


## 8. Geographic Revenue Concentration
Examining domestic (UK) vs international market distribution.


In [ ]:
country_share = eda.get_country_breakdown(8)
plt.figure(figsize=(8, 8))
plt.pie(country_share['Revenue'], labels=country_share['Country'], autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
plt.title('Geographic Revenue Distribution (Top Markets)', fontsize=14, weight='bold')
plt.show()
